# 13 Redshift space

A survey measures redshifts, not distances, so each galaxy is displaced
along the line of sight by its peculiar velocity:
$s = x + v_\parallel (1 + z) / H(z)$. At $z = 0$ that is
$s = x + v_\parallel / (100\ \mathrm{km\,s^{-1}})$ in $h^{-1}\,\mathrm{Mpc}$.
Clusters stretch into fingers along the line of sight and coherent infall
squashes walls across it. The labels stay the real-space T-web: the network
is asked for the true structure from the distorted tracers.

This notebook

1. builds redshift-space subhalo grids from the TNG300-1 catalog
   (`SubhaloVel`), with z as the line of sight, the same subhalos and the
   same mass-weighted cloud-in-cell grid as the real-space inputs;
2. checks them, and works out which axis the older `*_RSD.fvol` grids used;
3. shows real against redshift space;
4. scores the real-space model (the packaged one) on redshift-space inputs
   without retraining, as the baseline for a model trained in redshift space
   (`configs/research/rsd/C_mixed_smooth2ch_depth5_rsd.yaml`, run with
   notebook 05).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/deepvoid
!git clone -q https://github.com/skumagai3/deepvoid.git /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!git -C /content/deepvoid log --oneline -1

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from deepvoid import Catalog
from deepvoid.grid import cic
from deepvoid.io import save_volume
from deepvoid.observe import select_by_spacing, to_redshift_space
from deepvoid.sources.tng import read_subhalos

catalog = Catalog()
SPACINGS = (3, 5, 7, 10)
N = 512
LOS = 2                                   # z
subhalo_file = catalog.root / 'TNG' / 'TNG300-1_subhalos_snap99.npz'   # from notebook 10
out_dir = catalog.root / 'TNG' / 'rsd'
out_dir.mkdir(exist_ok=True)
subhalos = read_subhalos(subhalo_file)
assert subhalos.vel is not None, 'this catalog has no SubhaloVel; refetch it with notebook 10'
print(f'{len(subhalos):,} subhalos; line-of-sight velocity rms '
      f'{subhalos.vel[:, LOS].std():.0f} km/s')

## 1. Build the redshift-space grids

For each spacing: the same most massive subhalos as the real-space input,
moved along z, gridded the same way. Two checks on each grid:

- the real-space grid made here must match the training input (correlation 1);
- moving tracers only along z cannot change how much mass lies in each
  (x, y) column, so the z-projections of the real- and redshift-space grids
  must agree to float rounding.

In [ ]:
rows = {}
for s in SPACINGS:
    sample = select_by_spacing(subhalos, s)
    real = cic(sample, N)
    rsd = cic(to_redshift_space(sample, axis=LOS), N)
    train = catalog.load(f'tng300-3/subhalos/L{s}').data
    a, b = real.data.sum(axis=LOS), rsd.data.sum(axis=LOS)
    shift = sample.vel[:, LOS] / 100.0
    rows[f'L{s}'] = {
        'tracers': len(sample),
        'corr. with training input': np.corrcoef(real.data.ravel(), train.ravel())[0, 1],
        'z-projection max rel. diff.': float(np.abs(a - b).max() / a.max()),
        'rms shift (h^-1 Mpc)': float(shift.std()),
        'max |shift| (h^-1 Mpc)': float(np.abs(shift).max()),
    }
    rsd.meta.update(rsd=True, los_axis=LOS, source=str(subhalo_file.name))
    save_volume(rsd, out_dir / f'subhalos_L{s}_rsd_z.npz', dtype='float32')
    del sample, real, rsd, train
checks = pd.DataFrame(rows).T
print(checks.to_string())

## 2. The older redshift-space grids

`TNG/subs1_mass_Nm512_L*_RSD.fvol` were made before this repository, and the
code that made them is not in it, so their line of sight is unknown. The
projection test answers it: along the line of sight, the projection of the
redshift-space grid equals that of the real-space grid. Their correlation
with the new grids shows whether they are the same.

In [ ]:
rows = {}
for s in SPACINGS:
    old = catalog.load(f'tng300-3/subhalos/L{s}/rsd').data
    real = catalog.load(f'tng300-3/subhalos/L{s}').data
    new = catalog.load(f'tng300-3/subhalos/L{s}/rsd-z').data
    r = {f'projection corr., axis {"xyz"[k]}':
         np.corrcoef(old.sum(axis=k).ravel(), real.sum(axis=k).ravel())[0, 1] for k in range(3)}
    r['corr. with new rsd-z grid'] = np.corrcoef(old.ravel(), new.ravel())[0, 1]
    rows[f'L{s}'] = r
    del old, real, new
print(pd.DataFrame(rows).T.round(6).to_string())

## 3. Real against redshift space

A slab of the box seen side on, with z (the line of sight) vertical. Both
fields are smoothed with a $1\,h^{-1}\,\mathrm{Mpc}$ Gaussian and averaged over
8 cells in y so the tracers show.

In [ ]:
from scipy import ndimage
FIGS = Path('/content/drive/MyDrive/deepvoid_runs/redshift_space_figures')
FIGS.mkdir(exist_ok=True)
Y0, DY = 256, 8
cell = 205.0 / N

def side_view(field):
    f = ndimage.gaussian_filter(field / field.mean(), 1.0 / cell, mode='wrap')
    return np.log10(1e-2 + f[:, Y0:Y0 + DY, :].mean(axis=1))

fig, axes = plt.subplots(2, 4, figsize=(20, 10.5))
for c, s in enumerate(SPACINGS):
    for r, (name, label) in enumerate(((f'tng300-3/subhalos/L{s}', 'real space'),
                                       (f'tng300-3/subhalos/L{s}/rsd-z', 'redshift space'))):
        img = side_view(catalog.load(name).data)
        ax = axes[r, c]
        ax.imshow(img.T, origin='lower', extent=[0, 205, 0, 205], cmap='magma',
                  vmin=np.percentile(img, 5), vmax=np.percentile(img, 99.8))
        ax.set_title(f'$\\lambda = {s}$, {label}')
        ax.set_xlabel('x ($h^{-1}$ Mpc)')
        ax.set_ylabel('z, line of sight ($h^{-1}$ Mpc)')
fig.tight_layout()
plt.savefig(FIGS / 'real_vs_redshift_space.png', dpi=110, bbox_inches='tight')
plt.show()

## 4. The real-space model on redshift-space inputs

No retraining: the packaged model (notebook 11) predicts the whole box from
the redshift-space grids, scored on the run's test slab against the
real-space T-web labels. The real-space scores of the same slab are in the
first column for comparison. This is the gap a model trained in redshift
space has to close.

In [ ]:
from deepvoid.evaluate import confusion_matrix, scores
from deepvoid.pretrained import Pretrained
from deepvoid.split import Region

MODEL = '/content/drive/MyDrive/deepvoid_model_release'
RUN = Path('/content/drive/MyDrive/deepvoid_runs/subhalo_C_mixed_smooth2ch_depth5')
model = Pretrained.from_folder(MODEL)
test = Region(**json.loads((RUN / 'split.json').read_text())['test'])
truth = catalog.load('tng300-3/mask').data
sel = test.mask(N)
real_scores = json.loads((RUN / 'test_scores.json').read_text())
KEYS = ['mcc', 'void_mcc', 'void_f1', 'wall_f1', 'filament_f1', 'halo_f1']
results, slices = {}, {}
for s in SPACINGS:
    vol = catalog.load(f'tng300-3/subhalos/L{s}/rsd-z')
    pred = model.predict(vol.data, vol.box_size, float(s)).argmax(-1)
    cm = confusion_matrix(truth[sel], pred[sel])
    results[s] = {'scores': {k: scores(cm)[k] for k in KEYS}, 'confusion_matrix': cm.tolist()}
    slices[s] = pred[:, Y0, :]
    print(s, {k: round(v, 3) for k, v in results[s]['scores'].items()}, flush=True)
    del vol, pred
json.dump(results, open(FIGS.parent / 'rsd_real_space_model.json', 'w'), indent=1)

table = pd.concat({
    'real-space input': pd.DataFrame({s: {k: real_scores[f'tng300-3/subhalos/L{s}'][k] for k in KEYS}
                                      for s in SPACINGS}).T,
    'redshift-space input': pd.DataFrame({s: results[s]['scores'] for s in SPACINGS}).T}, axis=1)
print(table.round(3).to_string())

In [ ]:
from deepvoid.figures import CLASS_NAMES, MASK_COLORS
from matplotlib.colors import ListedColormap
cmap = ListedColormap(MASK_COLORS)

def frac(cm):
    cm = np.asarray(cm, float)
    return cm / np.maximum(cm.sum(1, keepdims=True), 1)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for c, s in enumerate(SPACINGS):
    t = frac(real_scores[f'tng300-3/subhalos/L{s}']['confusion_matrix'])
    r = frac(results[s]['confusion_matrix'])
    for row, (m, cm_, lo, hi, fmt, title) in enumerate((
            (t, 'Blues', 0, 1, '{:.2f}', 'real-space input'),
            (r, 'Blues', 0, 1, '{:.2f}', 'redshift-space input'),
            (r - t, 'RdBu_r', -0.15, 0.15, '{:+.3f}', 'redshift $-$ real'))):
        ax = axes[row, c]
        ax.imshow(m, cmap=cm_, vmin=lo, vmax=hi)
        for i in range(4):
            for j in range(4):
                ax.text(j, i, fmt.format(m[i, j]), ha='center', va='center', fontsize=9,
                        color='white' if abs(m[i, j]) > 0.6 * hi else 'black')
        ax.set_xticks(range(4), CLASS_NAMES, fontsize=8)
        ax.set_yticks(range(4), CLASS_NAMES, fontsize=8)
        ax.set_title(f'$\\lambda = {s}$: {title}')
for ax in axes[:, 0]:
    ax.set_ylabel('true')
for ax in axes[2]:
    ax.set_xlabel('predicted')
fig.tight_layout()
plt.savefig(FIGS / 'confusion_real_vs_rsd_input.png', dpi=110, bbox_inches='tight')
plt.show()

# Side view (z vertical) of truth and predictions from redshift-space input.
fig, axes = plt.subplots(1, 5, figsize=(25, 5.4))
axes[0].imshow(truth[:, Y0, :].T, origin='lower', extent=[0, 205, 0, 205], cmap=cmap,
               vmin=-0.5, vmax=3.5, interpolation='nearest')
axes[0].set_title('T-web labels (real space)')
for ax, s in zip(axes[1:], SPACINGS):
    ax.imshow(slices[s].T, origin='lower', extent=[0, 205, 0, 205], cmap=cmap,
              vmin=-0.5, vmax=3.5, interpolation='nearest')
    ax.set_title(f'prediction from redshift space, $\\lambda = {s}$')
for ax in axes:
    ax.axvspan(test.start * cell, test.stop * cell, color='w', alpha=0.15)
    ax.set_xlabel('x ($h^{-1}$ Mpc)')
axes[0].set_ylabel('z, line of sight ($h^{-1}$ Mpc)')
fig.tight_layout()
plt.savefig(FIGS / 'rsd_predictions_side_view.png', dpi=110, bbox_inches='tight')
plt.show()